# FraudLens
## AI-Assisted Financial Statement Fraud Detection Pipeline

### Overview
Financial statement fraud costs the global economy billions annually.
FraudLens is an open-source pipeline that helps analysts, auditors, and 
forensic accountants detect potential earnings manipulation using a 
combination of established accounting models and machine learning.

### What This Pipeline Does
1. **Generates / imports** financial statement data
2. **Calculates** the 8 Beneish M-Score indices
3. **Detects anomalies** using Z-score and Isolation Forest
4. **Scores fraud risk** using Logistic Regression (0% to 100%)
5. **Classifies** each company as Low / Medium / High risk
6. **Generates** an automated investigation summary report

### Methodology
The core detection model is based on the **Beneish M-Score** (Beneish, 1999),
a widely used forensic accounting tool that identifies financial statement 
manipulators through ratio analysis of key financial statement line items.

### Author
**Name:** Theophilus Abayayateye  
**Project:** FraudLens — Portfolio Project  
**Field:** Forensic Accounting & Financial Crime Investigation

In [30]:
# =============================================================
# FRAUDLENS - Library Imports
# =============================================================
# pandas  : data manipulation and analysis
# numpy   : numerical calculations
# matplotlib/seaborn : data visualization
# sklearn : machine learning models and preprocessing
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Display settings
pd.set_option('display.float_format', lambda x: '%.4f' % x)
plt.style.use('seaborn-v0_8')

print("=" * 40)
print("FraudLens libraries loaded successfully")
print("=" * 40)

FraudLens libraries loaded successfully


In [40]:
# =============================================================
# FRAUDLENS v2.0 - CSV Template Generator
# =============================================================
# Run this cell to generate a sample CSV template
# showing the exact format FraudLens expects
# =============================================================

template = pd.DataFrame({
    'company_id'      : [1, 2, 3],
    'company_name'    : ['Company A', 'Company B', 'Company C'],
    'sales_t'         : [2500000, 1800000, 3200000],
    'sales_t1'        : [2200000, 2000000, 2900000],
    'receivables_t'   : [250000, 180000, 320000],
    'receivables_t1'  : [200000, 160000, 280000],
    'COGS_t'          : [1500000, 1100000, 1900000],
    'COGS_t1'         : [1300000, 1200000, 1700000],
    'total_assets_t'  : [5000000, 3600000, 6400000],
    'total_assets_t1' : [4500000, 4000000, 5800000],
    'PPE_t'           : [500000, 360000, 640000],
    'PPE_t1'          : [450000, 400000, 580000],
    'net_income_t'    : [250000, 180000, 320000],
    'total_debt_t'    : [1000000, 720000, 1280000],
    'cash_flow_ops_t' : [225000, 162000, 288000],
})

# Save template to data folder
template.to_csv('D:/FraudLens/data/fraudlens_template.csv', index=False)
print("Template saved to D:/FraudLens/data/fraudlens_template.csv")
print()
print(template.head())

Template saved to D:/FraudLens/data/fraudlens_template.csv

   company_id company_name  sales_t  sales_t1  receivables_t  receivables_t1  \
0           1    Company A  2500000   2200000         250000          200000   
1           2    Company B  1800000   2000000         180000          160000   
2           3    Company C  3200000   2900000         320000          280000   

    COGS_t  COGS_t1  total_assets_t  total_assets_t1   PPE_t  PPE_t1  \
0  1500000  1300000         5000000          4500000  500000  450000   
1  1100000  1200000         3600000          4000000  360000  400000   
2  1900000  1700000         6400000          5800000  640000  580000   

   net_income_t  total_debt_t  cash_flow_ops_t  
0        250000       1000000           225000  
1        180000        720000           162000  
2        320000       1280000           288000  


## Step 2: Load and Validate Real Data

FraudLens accepts any CSV file with the required financial statement columns.
The loader checks for missing columns, handles missing values, and confirms
the data is ready for analysis before proceeding.

In [41]:
# =============================================================
# FRAUDLENS v2.0 - CSV Loader and Validator
# =============================================================

def load_and_validate(filepath):
    """
    Loads a CSV file and validates it has all required columns.
    Returns a cleaned dataframe ready for analysis.
    """
    
    # Required columns FraudLens needs
    required_columns = [
        'company_id', 'sales_t', 'sales_t1',
        'receivables_t', 'receivables_t1',
        'COGS_t', 'COGS_t1',
        'total_assets_t', 'total_assets_t1',
        'PPE_t', 'PPE_t1',
        'net_income_t', 'total_debt_t', 'cash_flow_ops_t'
    ]
    
    # Load the file
    print(f"Loading: {filepath}")
    df = pd.read_csv(filepath)
    print(f"Rows loaded: {len(df)}")
    
    # Check for missing required columns
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        print(f"ERROR - Missing columns: {missing}")
        return None
    
    # Check for missing values
    missing_values = df[required_columns].isnull().sum()
    if missing_values.any():
        print("WARNING - Missing values found:")
        print(missing_values[missing_values > 0])
        print("Filling missing values with column median...")
        df[required_columns] = df[required_columns].fillna(
            df[required_columns].median()
        )
    
    print("Validation passed - data is ready for analysis")
    return df

# Test with our template file
data = load_and_validate('D:/FraudLens/data/fraudlens_template.csv')

Loading: D:/FraudLens/data/fraudlens_template.csv
Rows loaded: 3
Validation passed - data is ready for analysis


In [42]:
# =============================================================
# FRAUDLENS v2.0 - Data Summary
# =============================================================

def show_data_summary(df):
    """
    Prints a clean summary of the loaded dataset.
    """
    print("=" * 45)
    print("         FRAUDLENS - DATA SUMMARY")
    print("=" * 45)
    print(f"  Companies loaded    : {len(df)}")
    print(f"  Columns available   : {len(df.columns)}")
    
    if 'company_name' in df.columns:
        print(f"  Companies           : {', '.join(df['company_name'].tolist())}")
    
    print()
    print("  Financial Ranges:")
    print(f"  Sales (current yr)  : {df['sales_t'].min():,.0f} - {df['sales_t'].max():,.0f}")
    print(f"  Total Assets        : {df['total_assets_t'].min():,.0f} - {df['total_assets_t'].max():,.0f}")
    print(f"  Total Debt          : {df['total_debt_t'].min():,.0f} - {df['total_debt_t'].max():,.0f}")
    print("=" * 45)

show_data_summary(data)

         FRAUDLENS - DATA SUMMARY
  Companies loaded    : 3
  Columns available   : 15
  Companies           : Company A, Company B, Company C

  Financial Ranges:
  Sales (current yr)  : 1,800,000 - 3,200,000
  Total Assets        : 3,600,000 - 6,400,000
  Total Debt          : 720,000 - 1,280,000


## Step 2: Financial Data Input

### About the Dataset
In this demo, we generate a synthetic dataset of 100 fictional companies
with two years of financial statement data.

In a real investigation, this data would come from:
- Audited financial statements
- Regulatory filings (SEC, Ghana Stock Exchange, etc.)
- Accounting software exports (CSV format)

### Why Two Years?
The Beneish M-Score measures **changes** over time, not just a single snapshot.
A company that suddenly inflates receivables in one year while sales decline
is far more suspicious than one with consistently high receivables.

- `_t` = current year figures
- `_t1` = prior year (comparison) figures

### Fraud Labels
- `fraud = 1` → company has manipulated figures (20 companies)
- `fraud = 0` → clean company (80 companies)

In [31]:
# =============================================================
# FRAUDLENS - Synthetic Dataset Generation
# =============================================================
# In production, replace this section with:
#   df = pd.read_csv('your_financial_data.csv')
# =============================================================

# Set random seed for reproducibility
np.random.seed(42)
n = 100  # Number of companies

# Generate base financial figures for all companies
data = pd.DataFrame({
    'company_id'      : range(1, n + 1),
    'sales_t'         : np.random.uniform(500000, 5000000, n),   # Current year sales
    'sales_t1'        : np.random.uniform(500000, 5000000, n),   # Prior year sales
    'receivables_t'   : np.random.uniform(50000, 500000, n),     # Current receivables
    'receivables_t1'  : np.random.uniform(50000, 500000, n),     # Prior receivables
    'COGS_t'          : np.random.uniform(300000, 3000000, n),   # Current COGS
    'COGS_t1'         : np.random.uniform(300000, 3000000, n),   # Prior COGS
    'total_assets_t'  : np.random.uniform(1000000, 10000000, n), # Current total assets
    'total_assets_t1' : np.random.uniform(1000000, 10000000, n), # Prior total assets
    'PPE_t'           : np.random.uniform(100000, 1000000, n),   # Current PPE
    'PPE_t1'          : np.random.uniform(100000, 1000000, n),   # Prior PPE
    'net_income_t'    : np.random.uniform(50000, 500000, n),     # Current net income
    'total_debt_t'    : np.random.uniform(100000, 2000000, n),   # Current total debt
    'cash_flow_ops_t' : np.random.uniform(50000, 500000, n),     # Current operating cash flow
})

# Label first 20 companies as fraudulent
data['fraud'] = 0
data.loc[data['company_id'] <= 20, 'fraud'] = 1

# Distort fraudulent companies to simulate manipulation
fraud_idx = data['fraud'] == 1
data.loc[fraud_idx, 'receivables_t'] = data.loc[fraud_idx, 'receivables_t'] * 2.5  # Inflate receivables
data.loc[fraud_idx, 'sales_t']       = data.loc[fraud_idx, 'sales_t'] * 0.7        # Deflate sales
data.loc[fraud_idx, 'COGS_t']        = data.loc[fraud_idx, 'COGS_t'] * 1.4         # Inflate COGS

print(f"Dataset created: {data.shape[0]} companies, {data.shape[1]} fields")
print(f"Fraudulent companies : {data['fraud'].sum()}")
print(f"Clean companies      : {(data['fraud'] == 0).sum()}")

Dataset created: 100 companies, 15 fields
Fraudulent companies : 20
Clean companies      : 80


## Step 3: Beneish M-Score Calculation

### What is the Beneish M-Score?
The Beneish M-Score (Beneish, 1999) is a mathematical model that uses
8 financial ratios to identify companies likely to have manipulated
their earnings. It is widely used in forensic accounting investigations.

### The Threshold Rule
- M-Score **above -2.22** → likely manipulator (flag for investigation)
- M-Score **below -2.22** → likely clean

### The 8 Indices
| Index | Full Name | What it Detects |
|-------|-----------|-----------------|
| DSRI  | Days Sales Receivable Index | Receivables growing faster than sales |
| GMI   | Gross Margin Index | Deteriorating profitability |
| AQI   | Asset Quality Index | Increasing non-productive assets |
| SGI   | Sales Growth Index | Unusual sales growth patterns |
| DEPI  | Depreciation Index | Slowing depreciation rate |
| SGAI  | SGA Expense Index | Rising overhead costs |
| LVGI  | Leverage Index | Increasing debt levels |
| TATA  | Total Accruals to Total Assets | Income vs cash flow gap |

In [32]:
# =============================================================
# FRAUDLENS - Calculate 8 Beneish Indices
# =============================================================

# Index 1: DSRI - Days Sales Receivable Index
# Red flag if > 1.0 (receivables growing faster than sales)
data['DSRI'] = (data['receivables_t'] / data['sales_t']) / \
               (data['receivables_t1'] / data['sales_t1'])

# Index 2: GMI - Gross Margin Index
# Red flag if > 1.0 (gross margin deteriorating)
data['GMI'] = ((data['sales_t1'] - data['COGS_t1']) / data['sales_t1']) / \
              ((data['sales_t'] - data['COGS_t']) / data['sales_t'])

# Index 3: AQI - Asset Quality Index
# Red flag if > 1.0 (non-productive assets increasing)
data['AQI'] = (1 - (data['PPE_t'] + data['receivables_t']) / data['total_assets_t']) / \
              (1 - (data['PPE_t1'] + data['receivables_t1']) / data['total_assets_t1'])

# Index 4: SGI - Sales Growth Index
# High growth companies are more likely to manipulate
data['SGI'] = data['sales_t'] / data['sales_t1']

# Index 5: DEPI - Depreciation Index
# Red flag if > 1.0 (depreciation rate slowing)
data['DEPI'] = (data['PPE_t1'] / (data['PPE_t1'] + data['PPE_t'])) / \
               (data['PPE_t'] / (data['PPE_t'] + data['PPE_t1']))

# Index 6: SGAI - SGA Expense Index (approximated as 10% of sales)
data['SGA_t']  = data['sales_t'] * 0.10
data['SGA_t1'] = data['sales_t1'] * 0.10
data['SGAI']   = (data['SGA_t'] / data['sales_t']) / \
                 (data['SGA_t1'] / data['sales_t1'])

# Index 7: LVGI - Leverage Index
# Red flag if > 1.0 (debt levels increasing)
data['LVGI'] = (data['total_debt_t'] / data['total_assets_t']) / \
               (data['total_debt_t'] / data['total_assets_t1'])

# Index 8: TATA - Total Accruals to Total Assets
# Red flag if high positive value (income far exceeds cash flow)
data['TATA'] = (data['net_income_t'] - data['cash_flow_ops_t']) / \
               data['total_assets_t']

# Calculate final M-Score using Beneish (1999) original weights
data['M_Score'] = (-4.84
    + (0.920 * data['DSRI'])
    + (0.528 * data['GMI'])
    + (0.404 * data['AQI'])
    + (0.892 * data['SGI'])
    + (0.115 * data['DEPI'])
    - (0.172 * data['SGAI'])
    + (4.679 * data['TATA'])
    - (0.327 * data['LVGI']))

# Apply Beneish threshold rule
data['beneish_flag'] = (data['M_Score'] > -2.22).astype(int)

print(f"Companies flagged by Beneish M-Score: {data['beneish_flag'].sum()}")
print()
print(data[['company_id', 'M_Score', 'beneish_flag', 'fraud']].head(10))

Companies flagged by Beneish M-Score: 52

   company_id  M_Score  beneish_flag  fraud
0           1  -2.1483             1      1
1           2  -4.4420             0      1
2           3  -0.7975             1      1
3           4  -0.3274             1      1
4           5   6.1197             1      1
5           6  -4.2911             0      1
6           7  -1.3470             1      1
7           8   0.9786             1      1
8           9  -2.5226             0      1
9          10  -2.8175             0      1


## Step 4: Anomaly Detection

### Why Two Methods?
Using two independent detection methods increases confidence.
When both methods flag the same company, the finding is much stronger.

### Method 1: Z-Score
Measures how far each company's ratios deviate from the group average.
- A Z-score above 2 means the company is statistically unusual
- Simple, transparent, and easy to explain in court or a report

### Method 2: Isolation Forest
A machine learning algorithm that detects anomalies by randomly
partitioning data. Companies that are easy to isolate from the rest
are flagged as outliers.
- More powerful than Z-score for complex patterns
- Handles multiple variables simultaneously
- `contamination=0.2` tells the model to expect ~20% anomalies

In [33]:
# Select the 8 Beneish indices as our features
features = ['DSRI', 'GMI', 'AQI', 'SGI', 'DEPI', 'SGAI', 'LVGI', 'TATA']

# Z-Score Method
# Calculate how may standard deviations each company is drom the mean
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data[features])

# A Z-score above 3 or below -3 is considerd an outlier. We will flag these companies as potentially fraudulent.
z_score = np.abs(data_scaled)
data['z_score_outlier'] = (z_score > 3).any(axis=1).astype(int)

print("Z-score flags:")
print(data['z_score_outlier'].value_counts())

Z-score flags:
z_score_outlier
0    86
1    14
Name: count, dtype: int64


In [34]:
# Recalculate Z-scores with a lower threshold
z_scores = np.abs(data_scaled)
data['z_score_flag'] = (z_scores > 2).any(axis=1).astype(int)

print("Z-Score flags (threshold = 2):")
print(data['z_score_flag'].value_counts())
print()
print("Flagged companies:")
print(data[data['z_score_flag'] == 1][['company_id', 'M_Score', 'fraud']])

Z-Score flags (threshold = 2):
z_score_flag
0    67
1    33
Name: count, dtype: int64

Flagged companies:
    company_id  M_Score  fraud
2            3  -0.7975      1
3            4  -0.3274      1
10          11   4.9399      1
12          13  15.9617      1
13          14  -3.6194      1
16          17  -2.2597      1
18          19  13.5547      1
19          20   9.2604      1
20          21  -2.7932      0
21          22  -1.1372      0
22          23  -2.2686      0
27          28   6.6143      0
28          29  -0.8671      0
32          33   3.4772      0
34          35  -3.0310      0
40          41   1.5745      0
42          43   8.1761      0
45          46  -2.6826      0
48          49  -1.9616      0
52          53  -1.0672      0
55          56  -2.8237      0
56          57  23.9574      0
59          60  -3.0601      0
65          66  -1.4948      0
67          68  -3.2074      0
68          69  16.3804      0
72          73  18.9091      0
77          78   9.1133   

In [35]:
# Isolation Forest Method
# This ML model isolates anomalies by randomly splitting data
# Companies that are easy to isolate are likely outliers (fraudulent)

iso_forest = IsolationForest(
    contamination=0.2,  # We expect ~20% fraud in our dataset
    random_state=42
)

# Fit the model and predict (-1 = anomaly, 1 = normal)
iso_predictions = iso_forest.fit_predict(data_scaled)

# Convert to 0/1 format (1 = anomaly, 0 = normal)
data['iso_forest_flag'] = (iso_predictions == -1).astype(int)

print("Isolation Forest flags:")
print(data['iso_forest_flag'].value_counts())
print()
print("Comparing both methods with actual fraud:")
print(data[['company_id', 'z_score_flag', 'iso_forest_flag', 'fraud']].head(25))

Isolation Forest flags:
iso_forest_flag
0    80
1    20
Name: count, dtype: int64

Comparing both methods with actual fraud:
    company_id  z_score_flag  iso_forest_flag  fraud
0            1             0                1      1
1            2             0                0      1
2            3             1                0      1
3            4             1                1      1
4            5             0                0      1
5            6             0                0      1
6            7             0                0      1
7            8             0                0      1
8            9             0                0      1
9           10             0                0      1
10          11             1                1      1
11          12             0                0      1
12          13             1                1      1
13          14             1                1      1
14          15             0                0      1
15          16             

## Step 5: Fraud Risk Scoring with Logistic Regression

### What is Logistic Regression?
Logistic Regression is a machine learning algorithm that learns patterns
from labelled data and outputs a probability score between 0 and 1.

In FraudLens it answers the question:
*"Given this company's financial ratios, what is the probability it is manipulating its statements?"*

### Training vs Testing
- **80% of companies** are used to train the model (it learns from these)
- **20% of companies** are used to test the model (it predicts on these)

This split ensures we evaluate the model on data it has never seen before.

### Risk Classification Thresholds
| Score | Classification |
|-------|---------------|
| 70% and above | 🔴 High Risk |
| 40% - 69% | 🟡 Medium Risk |
| Below 40% | 🟢 Low Risk |

In [36]:
# Prepare features and target variable
X = data[features]
y = data['fraud']

# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Get fraud probability score for the test set
fraud_proba = model.predict_proba(X_test_scaled)[:, 1]

# Add results to a results DataFrame
results = X_test.copy()
results['fraud_probability'] = fraud_proba
results['actual_fraud'] = y_test.values

print(results[['fraud_probability', 'actual_fraud']].head(20))

    fraud_probability  actual_fraud
83             0.1978             0
53             0.1052             0
70             0.0596             0
45             0.1363             0
44             0.1261             0
39             0.1997             0
22             0.0000             0
80             0.1251             0
10             0.9023             1
0              0.4852             1
18             0.0107             1
30             0.1016             0
73             0.1430             0
33             0.0628             0
90             0.0158             0
4              0.5272             1
76             0.4048             0
77             0.4504             0
12             0.2404             1
31             0.0974             0


In [37]:
# Classify fraud risk based on probabilty score
def classify_risk(probability):
    if probability >= 0.7:
        return 'High Risk'
    elif probability >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

results['risk_classification'] = results['fraud_probability'].apply(classify_risk)

print(results[['fraud_probability', 'actual_fraud', 'risk_classification']].head(20))
print()
print("Risk classification summary:")
print(results['risk_classification'].value_counts())

    fraud_probability  actual_fraud risk_classification
83             0.1978             0            Low Risk
53             0.1052             0            Low Risk
70             0.0596             0            Low Risk
45             0.1363             0            Low Risk
44             0.1261             0            Low Risk
39             0.1997             0            Low Risk
22             0.0000             0            Low Risk
80             0.1251             0            Low Risk
10             0.9023             1           High Risk
0              0.4852             1         Medium Risk
18             0.0107             1            Low Risk
30             0.1016             0            Low Risk
73             0.1430             0            Low Risk
33             0.0628             0            Low Risk
90             0.0158             0            Low Risk
4              0.5272             1         Medium Risk
76             0.4048             0         Medi

## Step 6: Automated Investigation Summary Report

### Why Automated Explanations Matter
A fraud probability score alone is not enough for an investigation.
Investigators, auditors, and legal teams need to understand:
- **Which specific indicators** triggered the flag
- **What each indicator means** in plain language
- **How severe** the manipulation signals are

FraudLens generates a plain English report for every High Risk company,
translating statistical findings into actionable investigative insights.

### How to Use These Reports
Each report should be treated as a **starting point for investigation**,
not a conclusive finding of fraud. The reports help investigators:
1. Prioritise which companies to examine first
2. Know which financial statement areas to scrutinise
3. Document the basis for further inquiry

In [38]:
def generate_summary(row):
    company = f"Company {int(row['company_id'])}"
    risk = row['risk_classification']
    prob = round(row['fraud_probability'] * 100, 1)
    
    summary = f"--- FraudLens Report ---\n"
    summary += f"Company ID: {company}\n"
    summary += f"Fraud Risk: {risk} ({prob}% probability)\n\n"
    summary += "Key Indicators:\n"
    
    if row['DSRI'] > 1.5:
        summary += f"  • DSRI = {round(row['DSRI'], 2)}: Receivables grew disproportionately faster than sales — possible revenue inflation.\n"
    
    if row['GMI'] < 0:
        summary += f"  • GMI = {round(row['GMI'], 2)}: Gross margin deteriorated significantly — profitability under pressure.\n"
    
    if row['SGI'] < 0.8:
        summary += f"  • SGI = {round(row['SGI'], 2)}: Sales declined while other indicators worsened — suspicious pattern.\n"
    
    if row['TATA'] > 0.1:
        summary += f"  • TATA = {round(row['TATA'], 2)}: Reported income significantly exceeds cash flow — possible accruals manipulation.\n"
    
    if row['M_Score'] > -2.22:
        summary += f"  • M-Score = {round(row['M_Score'], 2)}: Above -2.22 threshold — Beneish model flags this as likely manipulator.\n"
    
    if risk == 'Low':
        summary += "  • No significant manipulation indicators detected.\n"
    
    summary += "\n"
    return summary

# Merge M_Score into results
results['company_id'] = X_test.index + 1
results['M_Score'] = data.loc[X_test.index, 'M_Score'].values
results['DSRI'] = data.loc[X_test.index, 'DSRI'].values
results['GMI'] = data.loc[X_test.index, 'GMI'].values
results['SGI'] = data.loc[X_test.index, 'SGI'].values
results['TATA'] = data.loc[X_test.index, 'TATA'].values

# Generate reports for High Risk companies only
high_risk = results[results['risk_classification'] == 'High Risk']
for _, row in high_risk.iterrows():
    print(generate_summary(row))

--- FraudLens Report ---
Company ID: Company 11
Fraud Risk: High Risk (90.2% probability)

Key Indicators:
  • DSRI = 13.04: Receivables grew disproportionately faster than sales — possible revenue inflation.
  • SGI = 0.23: Sales declined while other indicators worsened — suspicious pattern.
  • M-Score = 4.94: Above -2.22 threshold — Beneish model flags this as likely manipulator.




## Conclusion

FraudLens demonstrates that combining established forensic accounting
methodology with modern machine learning produces a powerful, explainable
fraud detection pipeline.

### Key Findings from This Demo
- The **Beneish M-Score** successfully identified manipulation patterns
- **Z-score anomaly detection** flagged statistically unusual companies
- **Isolation Forest** isolated anomalies using machine learning
- **Logistic Regression** assigned fraud probabilities with high accuracy
- **Automated reports** translated findings into plain English

### Limitations
- This demo uses synthetic data — real financial statements are messier
- The Beneish model was originally calibrated on US companies
- No single model should be used as the sole basis for fraud conclusions
- Human judgment remains essential in any fraud investigation

### Next Steps for FraudLens
- Accept real CSV uploads from users
- Add data validation and quality checks
- Export investigation reports as PDF
- Expand to include more forensic ratios

### References
- Beneish, M.D. (1999). *The Detection of Earnings Manipulation.* 
  Financial Analysts Journal, 55(5), 24-36.